# IntelliGen Complaint Intelligence — Real Data & Cloud Training
This notebook is designed for VS Code/Jupyter locally or on the existing Azure ML compute instance. It uses the processed CFPB complaint dataset and records model evaluation/provenance.


In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
print('Training platform:', os.getenv('TRAINING_PLATFORM', 'local-notebook'))


## 1. Load the processed CFPB data


In [ ]:
import pandas as pd
data_path = ROOT / 'data' / 'processed' / 'cfpb_complaints.csv'
df = pd.read_csv(data_path)
display(df.head())
print('Rows:', len(df))
display(df['category'].value_counts())


## 2. Inspect complaint text and class balance
Use this section to discuss representativeness, mapped CFPB product categories and class imbalance before model training.


In [ ]:
print(df['message'].str.len().describe())
display(df.groupby('category')['message'].count().sort_values(ascending=False).rename('count'))


## 3. Train and evaluate the auditable baseline
The training function writes `complaint_classifier.joblib`, `metrics.json`, and `training_provenance.json`.


In [ ]:
from app.ai.training import train_classifier
os.environ.setdefault('TRAINING_PLATFORM', 'azure_ml_compute_instance' if os.getenv('AZURE_ML_COMPUTE') else 'local_notebook')
metrics = train_classifier(data_path, ROOT / 'artifacts')
{k: metrics[k] for k in ['dataset_rows','accuracy','macro_precision','macro_recall','macro_f1','weighted_f1']}


## 4. Inspect per-class results


In [ ]:
report = pd.DataFrame(metrics['classification_report']).T
display(report)


## 5. Test classification and XAI


In [ ]:
from app.ai.classifier import ComplaintClassifier
clf = ComplaintClassifier(ROOT / 'artifacts', data_path)
sample = 'There is an unauthorised transaction on my card and I need the charge investigated.'
prediction = clf.predict(sample)
explanation = clf.explain(sample, prediction['label'])
print(prediction)
print(explanation)


## 6. Training provenance
When this notebook/script is run on the Azure ML compute instance, set the workspace/compute environment variables so the artifact records cloud provenance.


In [ ]:
prov_path = ROOT / 'artifacts' / 'training_provenance.json'
print(json.dumps(json.loads(prov_path.read_text()), indent=2))


## Evaluation discussion prompts
- Compare accuracy with macro precision/recall/F1 rather than relying on accuracy alone.
- Inspect weak classes and likely label overlap.
- Discuss domain shift: CFPB complaints are US financial-service complaints and may not represent a UK commercial support environment.
- Compare the lightweight baseline with optional transformer enhancements in terms of performance, explainability, latency and environmental cost.
- Keep final customer decisions and outbound replies under human review.
